# Project 3, Part 1, Create and load refugees table

## Included Modules and Packages

In [1]:
import csv

import math
import numpy as np
import pandas as pd

import psycopg2

## Supporting code

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [4]:
cursor = connection.cursor()

In [5]:
def my_read_csv_file(file_name, limit):
    "read the csv file and print only the first limit rows"
    
    csv_file = open(file_name, "r")
    
    csv_data = csv.reader(csv_file)
    
    i = 0
    
    for row in csv_data:
        i += 1
        if i <= limit:
            print(row)
            
    print("\nPrinted ", min(limit, i), "lines of ", i, "total lines.")

## 3.1.1 Drop the staging table stage_1_refugees if it exists

In [6]:
connection.rollback()

query = """

drop table if exists stage_1_refugees;

"""

cursor.execute(query)

connection.commit()

## 3.1.2 Create the staging table stage_1_refugees

In [7]:
connection.rollback()

query = """

create table stage_1_refugees (
    stage_id serial,
    year varchar(100),
    coo_name varchar(100),
    coo varchar(100),
    coo_iso varchar(100),
    coa_name varchar(100),
    coa varchar(100),
    coa_iso varchar(100),
    refugees varchar(100),
    asylum_seekers varchar(100),
    returned_refugees varchar(100),
    idps varchar(100),
    returned_idps varchar(100),
    stateless varchar(100),
    ooc varchar(100),
    oip varchar(100),
    hst varchar(100)
);

"""

cursor.execute(query)

connection.commit()

## 3.1.3 Display the population.csv

In [8]:
my_read_csv_file("population.csv", limit=10)

['year', 'coo_name', 'coo', 'coo_iso', 'coa_name', 'coa', 'coa_iso', 'refugees', 'asylum_seekers', 'returned_refugees', 'idps', 'returned_idps', 'stateless', 'ooc', 'oip', 'hst']
['2010', 'Afghanistan', 'AFG', 'AFG', 'Afghanistan', 'AFG', 'AFG', '0', '0', '0', '351907', '3366', '0', '838250', '', '']
['2010', 'Iran (Islamic Rep. of)', 'IRN', 'IRN', 'Afghanistan', 'AFG', 'AFG', '30', '21', '0', '0', '0', '0', '0', '', '']
['2010', 'Iraq', 'IRQ', 'IRQ', 'Afghanistan', 'AFG', 'AFG', '6', '0', '0', '0', '0', '0', '0', '', '']
['2010', 'Pakistan', 'PAK', 'PAK', 'Afghanistan', 'AFG', 'AFG', '6398', '9', '0', '0', '0', '0', '0', '', '']
['2010', 'Egypt', 'ARE', 'EGY', 'Albania', 'ALB', 'ALB', '5', '0', '0', '0', '0', '0', '0', '', '']
['2010', 'China', 'CHI', 'CHN', 'Albania', 'ALB', 'ALB', '6', '0', '0', '0', '0', '0', '0', '', '']
['2010', 'Palestinian', 'GAZ', 'PSE', 'Albania', 'ALB', 'ALB', '5', '0', '0', '0', '0', '0', '0', '', '']
['2010', 'Iraq', 'IRQ', 'IRQ', 'Albania', 'ALB', 'ALB', 

## 3.1.4 Load the CSV file refugees.csv into the table stage_1_refugees

In [9]:
connection.rollback()

query = """

copy stage_1_refugees (year, coo_name, coo, coo_iso, coa_name, coa, coa_iso, refugees,
                       asylum_seekers, returned_refugees, idps, returned_idps, stateless, ooc, oip, hst)
from '/user/projects/project-3-shantiagung-midsberkeley/code/population.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

## 3.1.5 Verify the load of stage_1_refugees by querying the entire table

In [10]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stage_1_refugees
order by stage_id
limit 20;

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,stage_id,year,coo_name,coo,coo_iso,coa_name,coa,coa_iso,refugees,asylum_seekers,returned_refugees,idps,returned_idps,stateless,ooc,oip,hst
0,1,2010,Afghanistan,AFG,AFG,Afghanistan,AFG,AFG,0,0,0,351907,3366,0,838250,None,None
1,2,2010,Iran (Islamic Rep. of),IRN,IRN,Afghanistan,AFG,AFG,30,21,0,0,0,0,0,None,None
2,3,2010,Iraq,IRQ,IRQ,Afghanistan,AFG,AFG,6,0,0,0,0,0,0,None,None
3,4,2010,Pakistan,PAK,PAK,Afghanistan,AFG,AFG,6398,9,0,0,0,0,0,None,None
4,5,2010,Egypt,ARE,EGY,Albania,ALB,ALB,5,0,0,0,0,0,0,None,None
5,6,2010,China,CHI,CHN,Albania,ALB,ALB,6,0,0,0,0,0,0,None,None
6,7,2010,Palestinian,GAZ,PSE,Albania,ALB,ALB,5,0,0,0,0,0,0,None,None
7,8,2010,Iraq,IRQ,IRQ,Albania,ALB,ALB,5,0,0,0,0,0,0,None,None
8,9,2010,Serbia and Kosovo: S/RES/1244 (1999),SRB,SRB,Albania,ALB,ALB,49,20,0,0,0,0,0,None,None
9,10,2010,Türkiye,TUR,TUR,Albania,ALB,ALB,5,0,0,0,0,0,0,None,None
